In [ ]:
import os

vctk_root = "./DR-VCTK/"
esc_root = "./ESC-50-master/"

for root, dirs, files in os.walk(vctk_root):
    print("VCTK:", root, len(files))
    break

for root, dirs, files in os.walk(esc_root):
    print("ESC50:", root, len(files))
    break

In [ ]:
from pathlib import Path

vctk_files = list(Path(vctk_root).rglob("*.wav"))
esc_files = list(Path(esc_root).rglob("*.wav"))

print("VCTK wav files:", len(vctk_files))
print("ESC wav files:", len(esc_files))
print("Sample VCTK:", vctk_files[:5])
print("Sample ESC:", esc_files[:5])

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm

In [ ]:


# =========================================================
# CONFIG
# =========================================================
RANDOM_SEED = 42
TARGET_SR = 16000
TARGET_TRAIN_SAMPLES = 15000
# Your folders
VCTK_ROOT = Path("DR-VCTK/clean_trainset_wav_16k")
ESC_ROOT = Path("ESC-50-master")
ESC_AUDIO_DIR = ESC_ROOT / "audio"
ESC_META_CSV = ESC_ROOT / "meta" / "esc50.csv"

# Output folder
OUTPUT_ROOT = Path("my_speech_enhancement_dataset")

# How much clean data to use from VCTK
MAX_CLEAN_FILES = 15000   # change if you want more or less

# Split ratios
TRAIN_RATIO = 0.75
VAL_RATIO = 0.125
TEST_RATIO = 0.125

# SNR values
TRAIN_SNRS = [-5, 0, 5, 10, 15,20]
VAL_SNRS = [-5, 0, 5, 10, 15,20]
TEST_SNRS = [-5, 0, 5, 10, 15,20]

# Allowed ESC-50 categories that behave more like background / environmental noise
SELECTED_NOISE_CATEGORIES = [
    "rain",
    "wind",
    "sea_waves",
    "crackling_fire",
    "helicopter",
    "train",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "clock_tick",
    "siren",
]

# Audio constraints
MIN_DURATION_SEC = 1.0
MAX_DURATION_SEC = 8.0   # optional cap for consistency

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =========================================================
# HELPERS
# =========================================================
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def load_audio(path: Path, sr: int = TARGET_SR):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio

def peak_normalize(audio: np.ndarray):
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

def trim_or_keep(audio: np.ndarray, sr: int, max_duration_sec: float):
    max_len = int(sr * max_duration_sec)
    if len(audio) > max_len:
        audio = audio[:max_len]
    return audio

def is_valid_audio(audio: np.ndarray, sr: int, min_duration_sec: float):
    return len(audio) >= int(sr * min_duration_sec)

def match_length(noise: np.ndarray, target_len: int):
    if len(noise) < target_len:
        reps = int(np.ceil(target_len / len(noise)))
        noise = np.tile(noise, reps)
    return noise[:target_len]

def mix_audio(clean: np.ndarray, noise: np.ndarray, snr_db: float):
    noise = match_length(noise, len(clean))

    clean_power = np.mean(clean ** 2) + 1e-12
    noise_power = np.mean(noise ** 2) + 1e-12

    target_noise_power = clean_power / (10 ** (snr_db / 10.0))
    scale = np.sqrt(target_noise_power / noise_power)
    noise_scaled = noise * scale

    noisy = clean + noise_scaled

    # Prevent clipping
    peak = np.max(np.abs(noisy))
    if peak > 0.99:
        noisy = noisy / peak * 0.99
        clean = clean / peak * 0.99

    return clean.astype(np.float32), noisy.astype(np.float32)

def split_list(items, train_ratio, val_ratio, test_ratio):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8
    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_items = items[:n_train]
    val_items = items[n_train:n_train + n_val]
    test_items = items[n_train + n_val:]

    return train_items, val_items, test_items

def extract_speaker_id(filepath: Path):
    # Example: p232_001.wav -> p232
    return filepath.stem.split("_")[0]

# =========================================================
# STEP 1: LOAD ESC-50 METADATA AND FILTER NOISE CLASSES
# =========================================================
print("=" * 70)
print("STEP 1: Reading ESC-50 metadata")
print("=" * 70)

esc_meta = pd.read_csv(ESC_META_CSV)
print("ESC-50 metadata rows:", len(esc_meta))
print("Available categories:", sorted(esc_meta["category"].unique()))

noise_meta = esc_meta[esc_meta["category"].isin(SELECTED_NOISE_CATEGORIES)].copy()
noise_meta["filepath"] = noise_meta["filename"].apply(lambda x: ESC_AUDIO_DIR / x)

print("\nSelected categories:")
print(sorted(noise_meta["category"].unique()))
print("Selected noise files:", len(noise_meta))

# =========================================================
# STEP 2: GET VCTK CLEAN FILES AND TAKE A SMALL SUBSET
# =========================================================
print("\n" + "=" * 70)
print("STEP 2: Loading VCTK clean files")
print("=" * 70)

all_clean_files = sorted(VCTK_ROOT.rglob("*.wav"))
print("Total VCTK files found:", len(all_clean_files))

# Shuffle once, then take smaller subset
random.shuffle(all_clean_files)
selected_clean_files = all_clean_files[:MAX_CLEAN_FILES]
selected_clean_files = sorted(selected_clean_files)

print("Selected clean files:", len(selected_clean_files))

# Optional: show speaker distribution
speaker_counts = {}
for f in selected_clean_files:
    spk = extract_speaker_id(f)
    speaker_counts[spk] = speaker_counts.get(spk, 0) + 1

print("Number of speakers in selected clean subset:", len(speaker_counts))
print("Sample speaker distribution:", list(speaker_counts.items())[:10])

# =========================================================
# STEP 3: FILTER VALID CLEAN FILES
# =========================================================
print("\n" + "=" * 70)
print("STEP 3: Validating selected clean files")
print("=" * 70)

valid_clean_files = []
for f in tqdm(selected_clean_files, desc="Checking clean"):
    try:
        audio = load_audio(f, sr=TARGET_SR)
        audio = trim_or_keep(audio, TARGET_SR, MAX_DURATION_SEC)
        if is_valid_audio(audio, TARGET_SR, MIN_DURATION_SEC):
            valid_clean_files.append(f)
    except Exception as e:
        print(f"Skipping clean file {f}: {e}")

print("Valid clean files:", len(valid_clean_files))

# =========================================================
# STEP 4: FILTER VALID NOISE FILES
# =========================================================
print("\n" + "=" * 70)
print("STEP 4: Validating selected ESC-50 noise files")
print("=" * 70)

valid_noise_rows = []
for _, row in tqdm(noise_meta.iterrows(), total=len(noise_meta), desc="Checking noise"):
    f = row["filepath"]
    try:
        audio = load_audio(f, sr=TARGET_SR)
        if is_valid_audio(audio, TARGET_SR, MIN_DURATION_SEC):
            valid_noise_rows.append(row)
    except Exception as e:
        print(f"Skipping noise file {f}: {e}")

valid_noise_meta = pd.DataFrame(valid_noise_rows).reset_index(drop=True)
print("Valid noise files:", len(valid_noise_meta))

# =========================================================
# STEP 5: SPLIT CLEAN FILES AND NOISE FILES
# =========================================================
print("\n" + "=" * 70)
print("STEP 5: Splitting clean and noise pools")
print("=" * 70)

random.shuffle(valid_clean_files)
clean_train, clean_val, clean_test = split_list(
    valid_clean_files, TRAIN_RATIO, VAL_RATIO, TEST_RATIO
)

noise_records = valid_noise_meta.to_dict("records")
random.shuffle(noise_records)
noise_train, noise_val, noise_test = split_list(
    noise_records, TRAIN_RATIO, VAL_RATIO, TEST_RATIO
)

print(f"Clean split -> train: {len(clean_train)}, val: {len(clean_val)}, test: {len(clean_test)}")
print(f"Noise split -> train: {len(noise_train)}, val: {len(noise_val)}, test: {len(noise_test)}")

# =========================================================
# STEP 6: GENERATE NOISY-CLEAN DATASET
# =========================================================
def generate_split(split_name, clean_files, noise_pool, snr_list, target_size=None):
    split_root = OUTPUT_ROOT / split_name
    clean_out_dir = split_root / "clean"
    noisy_out_dir = split_root / "noisy"

    ensure_dir(clean_out_dir)
    ensure_dir(noisy_out_dir)

    rows = []
    count = 0

    # decide how many times to loop
    if target_size is not None:
        total_needed = target_size
    else:
        total_needed = len(clean_files)

    idx = 0

    while count < total_needed:
        clean_path = clean_files[idx % len(clean_files)]

        try:
            clean_audio = load_audio(clean_path, sr=TARGET_SR)
            clean_audio = trim_or_keep(clean_audio, TARGET_SR, MAX_DURATION_SEC)
            clean_audio = peak_normalize(clean_audio)

            if not is_valid_audio(clean_audio, TARGET_SR, MIN_DURATION_SEC):
                idx += 1
                continue

            # random noise + snr each time
            noise_row = random.choice(noise_pool)
            noise_path = Path(noise_row["filepath"])
            noise_cat = noise_row["category"]

            noise_audio = load_audio(noise_path, sr=TARGET_SR)
            noise_audio = peak_normalize(noise_audio)

            snr_db = random.choice(snr_list)

            clean_mix, noisy_mix = mix_audio(clean_audio, noise_audio, snr_db)

            file_id = f"{count:05d}"

            clean_save = clean_out_dir / f"{file_id}.wav"
            noisy_save = noisy_out_dir / f"{file_id}.wav"

            sf.write(clean_save, clean_mix, TARGET_SR)
            sf.write(noisy_save, noisy_mix, TARGET_SR)

            rows.append({
                "id": file_id,
                "split": split_name,
                "speaker_id": extract_speaker_id(clean_path),
                "clean_path": str(clean_save),
                "noisy_path": str(noisy_save),
                "clean_source": str(clean_path),
                "noise_source": str(noise_path),
                "noise_category": noise_cat,
                "snr_db": snr_db,
                "duration_sec": round(len(clean_mix) / TARGET_SR, 3)
            })

            count += 1

        except Exception as e:
            print(f"Skipping sample {clean_path}: {e}")

        idx += 1

    df = pd.DataFrame(rows)
    df.to_csv(split_root / "metadata.csv", index=False)
    print(f"{split_name}: saved {len(df)} pairs")
print("\n" + "=" * 70)
print("STEP 6: Creating final paired dataset")
print("=" * 70)

generate_split("train", clean_train, noise_train, TRAIN_SNRS, target_size=1500)
generate_split("val", clean_val, noise_val, VAL_SNRS)
generate_split("test", clean_test, noise_test, TEST_SNRS)

# =========================================================
# DONE
# =========================================================
print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print("Dataset created at:", OUTPUT_ROOT.resolve())
print("\nFolder structure:")
print(f"{OUTPUT_ROOT}/train/clean")
print(f"{OUTPUT_ROOT}/train/noisy")
print(f"{OUTPUT_ROOT}/train/metadata.csv")
print(f"{OUTPUT_ROOT}/val/clean")
print(f"{OUTPUT_ROOT}/val/noisy")
print(f"{OUTPUT_ROOT}/val/metadata.csv")
print(f"{OUTPUT_ROOT}/test/clean")
print(f"{OUTPUT_ROOT}/test/noisy")
print(f"{OUTPUT_ROOT}/test/metadata.csv")

In [ ]:

from pathlib import Path
import random
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================
RANDOM_SEED = 42
TARGET_SR = 16000

# Requested final dataset sizes
TARGET_TRAIN_SAMPLES = 15000
TARGET_VAL_SAMPLES = 2000
TARGET_TEST_SAMPLES = 2000

MIN_DURATION_SEC = 1.0
MAX_DURATION_SEC = 8.0

VCTK_ROOTS = [
    Path("DR-VCTK/clean_trainset_wav_16k"),
    Path("DR-VCTK/clean_testset_wav_16k"),
    Path("VCTK-Corpus/wav16"),
]

ESC_ROOT = Path("ESC-50-master")
ESC_AUDIO_DIR = ESC_ROOT / "audio"
ESC_META_CSV = ESC_ROOT / "meta" / "esc50.csv"

OUTPUT_ROOT = Path("my_speech_enhancement_dataset_v3")

TRAIN_SNRS = [-5, 0, 5, 10, 15, 20]
VAL_SNRS = [-5, 0, 5, 10, 15, 20]
TEST_SNRS = [-5, 0, 5, 10, 15, 20]

SELECTED_NOISE_CATEGORIES = [
    "rain", "wind", "sea_waves", "crackling_fire", "helicopter",
    "train", "engine", "vacuum_cleaner", "washing_machine",
    "clock_tick", "siren",
]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =========================================================
# HELPERS
# =========================================================
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def load_audio(path: Path, sr: int = TARGET_SR):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio

def peak_normalize(audio: np.ndarray):
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

def trim_audio(audio: np.ndarray, sr: int, max_duration_sec: float):
    max_len = int(sr * max_duration_sec)
    if len(audio) > max_len:
        start = 0
        if len(audio) > max_len:
            start = np.random.randint(0, len(audio) - max_len + 1)
        audio = audio[start:start + max_len]
    return audio

def is_valid_audio(audio: np.ndarray, sr: int, min_duration_sec: float):
    return len(audio) >= int(sr * min_duration_sec)

def match_length(noise: np.ndarray, target_len: int):
    if len(noise) < target_len:
        reps = int(np.ceil(target_len / len(noise)))
        noise = np.tile(noise, reps)
    elif len(noise) > target_len:
        start = np.random.randint(0, len(noise) - target_len + 1)
        noise = noise[start:start + target_len]
    return noise[:target_len]

def random_gain(audio: np.ndarray, lo=0.85, hi=1.15):
    return audio * np.random.uniform(lo, hi)

def mix_audio(clean: np.ndarray, noise: np.ndarray, snr_db: float):
    noise = match_length(noise, len(clean))

    clean_power = np.mean(clean ** 2) + 1e-12
    noise_power = np.mean(noise ** 2) + 1e-12

    target_noise_power = clean_power / (10 ** (snr_db / 10.0))
    scale = np.sqrt(target_noise_power / noise_power)
    noise_scaled = noise * scale

    noisy = clean + noise_scaled

    peak = np.max(np.abs(noisy))
    if peak > 0.99:
        noisy = noisy / peak * 0.99
        clean = clean / peak * 0.99

    return clean.astype(np.float32), noisy.astype(np.float32)

def split_list(items, train_ratio, val_ratio, test_ratio):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8
    n = len(items)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_items = items[:n_train]
    val_items = items[n_train:n_train + n_val]
    test_items = items[n_train + n_val:]
    return train_items, val_items, test_items

def extract_speaker_id(filepath: Path):
    stem = filepath.stem
    return stem.split("_")[0] if "_" in stem else filepath.parent.name

def collect_clean_files(roots):
    clean_files = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for f in root.rglob("*.wav"):
            rf = f.resolve()
            if rf not in seen:
                seen.add(rf)
                clean_files.append(f)
    return sorted(clean_files)

# =========================================================
# STEP 1: LOAD NOISE METADATA
# =========================================================
print("=" * 70)
print("STEP 1: Reading ESC-50 metadata")
print("=" * 70)

esc_meta = pd.read_csv(ESC_META_CSV)
noise_meta = esc_meta[esc_meta["category"].isin(SELECTED_NOISE_CATEGORIES)].copy()
noise_meta["filepath"] = noise_meta["filename"].apply(lambda x: ESC_AUDIO_DIR / x)
print("Selected noise rows:", len(noise_meta))
print("Categories:", sorted(noise_meta["category"].unique()))

# =========================================================
# STEP 2: COLLECT CLEAN FILES FROM ALL ROOTS
# =========================================================
print("\n" + "=" * 70)
print("STEP 2: Collecting VCTK clean files from all roots")
print("=" * 70)

all_clean_files = collect_clean_files(VCTK_ROOTS)
print("Total clean wav files found:", len(all_clean_files))
for root in VCTK_ROOTS:
    print(f"  - {root}: {'FOUND' if root.exists() else 'MISSING'}")

# =========================================================
# STEP 3: VALIDATE CLEAN
# =========================================================
print("\n" + "=" * 70)
print("STEP 3: Validating clean files")
print("=" * 70)

valid_clean_files = []
for f in tqdm(all_clean_files, desc="Checking clean"):
    try:
        audio = load_audio(f, sr=TARGET_SR)
        audio = trim_audio(audio, TARGET_SR, MAX_DURATION_SEC)
        if is_valid_audio(audio, TARGET_SR, MIN_DURATION_SEC):
            valid_clean_files.append(f)
    except Exception as e:
        print(f"Skipping clean file {f}: {e}")

print("Valid clean files:", len(valid_clean_files))
if len(valid_clean_files) == 0:
    raise RuntimeError("No valid clean files found.")

# =========================================================
# STEP 4: VALIDATE NOISE
# =========================================================
print("\n" + "=" * 70)
print("STEP 4: Validating noise files")
print("=" * 70)

valid_noise_rows = []
for _, row in tqdm(noise_meta.iterrows(), total=len(noise_meta), desc="Checking noise"):
    try:
        audio = load_audio(Path(row["filepath"]), sr=TARGET_SR)
        if is_valid_audio(audio, TARGET_SR, MIN_DURATION_SEC):
            valid_noise_rows.append(row)
    except Exception as e:
        print(f"Skipping noise file {row['filepath']}: {e}")

valid_noise_meta = pd.DataFrame(valid_noise_rows).reset_index(drop=True)
print("Valid noise files:", len(valid_noise_meta))
if len(valid_noise_meta) == 0:
    raise RuntimeError("No valid noise files found.")

# =========================================================
# STEP 5: SPLIT CLEAN/NOISE POOLS
# =========================================================
print("\n" + "=" * 70)
print("STEP 5: Splitting clean/noise pools")
print("=" * 70)

random.shuffle(valid_clean_files)
clean_train, clean_val, clean_test = split_list(valid_clean_files, 0.75, 0.125, 0.125)

noise_records = valid_noise_meta.to_dict("records")
random.shuffle(noise_records)
noise_train, noise_val, noise_test = split_list(noise_records, 0.75, 0.125, 0.125)

print(f"Clean split -> train: {len(clean_train)}, val: {len(clean_val)}, test: {len(clean_test)}")
print(f"Noise split -> train: {len(noise_train)}, val: {len(noise_val)}, test: {len(noise_test)}")

# =========================================================
# STEP 6: FIXED FINAL OUTPUT SIZES
# =========================================================
train_target = TARGET_TRAIN_SAMPLES
val_target = TARGET_VAL_SAMPLES
test_target = TARGET_TEST_SAMPLES

print("\nFinal target sizes:")
print(f"  train: {train_target}")
print(f"  val  : {val_target}")
print(f"  test : {test_target}")

# =========================================================
# STEP 7: GENERATE DATASET
# =========================================================
def generate_split(split_name, clean_files, noise_pool, snr_list, target_size):
    split_root = OUTPUT_ROOT / split_name
    clean_out_dir = split_root / "clean"
    noisy_out_dir = split_root / "noisy"
    ensure_dir(clean_out_dir)
    ensure_dir(noisy_out_dir)

    rows = []
    count = 0
    idx = 0

    while count < target_size:
        clean_path = clean_files[idx % len(clean_files)]
        try:
            clean_audio = load_audio(clean_path, sr=TARGET_SR)
            clean_audio = trim_audio(clean_audio, TARGET_SR, MAX_DURATION_SEC)
            clean_audio = peak_normalize(clean_audio)
            clean_audio = random_gain(clean_audio, 0.9, 1.1)

            if not is_valid_audio(clean_audio, TARGET_SR, MIN_DURATION_SEC):
                idx += 1
                continue

            noise_row = random.choice(noise_pool)
            noise_path = Path(noise_row["filepath"])
            noise_cat = noise_row["category"]

            noise_audio = load_audio(noise_path, sr=TARGET_SR)
            noise_audio = peak_normalize(noise_audio)
            noise_audio = random_gain(noise_audio, 0.85, 1.15)

            snr_db = random.choice(snr_list)
            clean_mix, noisy_mix = mix_audio(clean_audio, noise_audio, snr_db)

            file_id = f"{count:05d}"
            clean_save = clean_out_dir / f"{file_id}.wav"
            noisy_save = noisy_out_dir / f"{file_id}.wav"

            sf.write(clean_save, clean_mix, TARGET_SR)
            sf.write(noisy_save, noisy_mix, TARGET_SR)

            rows.append({
                "id": file_id,
                "split": split_name,
                "speaker_id": extract_speaker_id(clean_path),
                "clean_path": str(clean_save),
                "noisy_path": str(noisy_save),
                "clean_source": str(clean_path),
                "noise_source": str(noise_path),
                "noise_category": noise_cat,
                "snr_db": snr_db,
                "duration_sec": round(len(clean_mix) / TARGET_SR, 3),
            })
            count += 1
        except Exception as e:
            print(f"Skipping sample {clean_path}: {e}")
        idx += 1

    df = pd.DataFrame(rows)
    df.to_csv(split_root / "metadata.csv", index=False)
    print(f"{split_name}: saved {len(df)} pairs")

print("\n" + "=" * 70)
print("STEP 7: Writing dataset")
print("=" * 70)

generate_split("train", clean_train if clean_train else valid_clean_files, noise_train if noise_train else noise_records, TRAIN_SNRS, train_target)
generate_split("val", clean_val if clean_val else valid_clean_files, noise_val if noise_val else noise_records, VAL_SNRS, val_target)
generate_split("test", clean_test if clean_test else valid_clean_files, noise_test if noise_test else noise_records, TEST_SNRS, test_target)

# =========================================================
# STEP 8: GLOBAL MANIFEST
# =========================================================
all_meta = []
for split in ["train", "val", "test"]:
    meta_path = OUTPUT_ROOT / split / "metadata.csv"
    if meta_path.exists() and meta_path.stat().st_size > 0:
        try:
            df = pd.read_csv(meta_path)
            if not df.empty:
                all_meta.append(df)
        except pd.errors.EmptyDataError:
            print(f"Skipping empty metadata file: {meta_path}")

if all_meta:
    manifest = pd.concat(all_meta, ignore_index=True)
    manifest.to_csv(OUTPUT_ROOT / "metadata_all.csv", index=False)
    print("Combined metadata saved:", OUTPUT_ROOT / "metadata_all.csv")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print("Dataset created at:", OUTPUT_ROOT.resolve())
